> **Note**: This notebook has been upgraded to support general Waste Detection. It dynamically generates colors for any number of classes and reads the central `configs/classes.yaml` for high-level waste group mapping.

# 🚀 Waste Detection — Inference & Deployment (Notebook 08)

### Overview
This notebook simulates the production deployment environment. It takes the trained `best.pt` model and runs inference on unseen images, outputting structured JSON and CSV results.

### New Features Added
- **Dynamic Color Palette**: Automatically generates distinct colors for ANY number of classes (solves the 8-class limit).
- **Waste Group Mapping**: Maps detailed TACO classes to high-level waste groups using the central `configs/classes.yaml`.
- **Improved JSON Export**: Now includes `[x1, y1, x2, y2]` bounding box formats required by modern backends.

### Pipeline Position
```
NB 06 (Training) → [models/best.pt] → NB 08 (THIS) → [predictions/]
```

## 1. Environment Setup & Library Installation

In [ ]:
!pip install -q ultralytics rich pyyaml pandas opencv-python matplotlib pillow

In [ ]:
import os
import json
import time
import shutil
from pathlib import Path
from typing import Dict, List, Tuple, Any, Union, Optional
from collections import defaultdict

import yaml
import torch
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from ultralytics import YOLO

console = Console()
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

## 2. Path Configuration & Asset Loading

In [ ]:
# ==========================================
# Paths
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/PlasticSense_AI')
MODELS_DIR = PROJECT_ROOT / 'models'
BEST_PT_PATH = MODELS_DIR / 'best.pt'
CONFIGS_DIR = PROJECT_ROOT / 'configs'
CLASSES_YAML = CONFIGS_DIR / 'classes.yaml'

# Input Images (using test set for demonstration)
TEST_IMAGES = PROJECT_ROOT / 'datasets' / 'taco_yolo_augmented' / 'images' / 'test'

# Outputs
INFERENCE_DIR = PROJECT_ROOT / 'results' / 'inference'
PRED_IMAGES_DIR = INFERENCE_DIR / 'images'
PRED_JSON_DIR = INFERENCE_DIR / 'json'
PRED_CSV_DIR = INFERENCE_DIR / 'csv'

for d in [PRED_IMAGES_DIR, PRED_JSON_DIR, PRED_CSV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ==========================================
# Load Central Configuration
# ==========================================
waste_groups = {}
if CLASSES_YAML.exists():
    with open(CLASSES_YAML, 'r') as f:
        config = yaml.safe_load(f)
    for cat in config.get('categories', []):
        waste_groups[cat['yolo_id']] = cat.get('waste_group', 'unknown')
    console.print(f"[green]✔ Loaded central classes.yaml[/green]")
else:
    console.print("[yellow]⚠ classes.yaml not found. Run Notebook 03 to generate it. High-level groups will be 'unknown'.[/yellow]")

## 3. Load Trained Model & Determine Classes

In [ ]:
if not BEST_PT_PATH.exists():
    raise FileNotFoundError(f"Model missing: {BEST_PT_PATH}. Run Notebook 06.")

console.print(f"[cyan]Loading model from {BEST_PT_PATH}...[/cyan]")
model = YOLO(str(BEST_PT_PATH))

# Load classes dynamically from model
class_names = {int(k): v for k, v in model.names.items()}
num_classes = len(class_names)
console.print(f"[green]✔ Model loaded successfully ({num_classes} classes detected)[/green]")

## 4. Dynamic Color Palette
Generate distinct BGR colors for ANY number of classes using HSV interpolation.

In [ ]:
# ==========================================
# Dynamic Color Generation
# ==========================================
def generate_class_colors(num_classes: int) -> Dict[int, Tuple[int, int, int]]:
    """Generates visually distinct BGR colors for N classes."""
    np.random.seed(42)
    colors_bgr = {}
    for cls_id in range(num_classes):
        hue = int(cls_id * 180 / max(num_classes, 1)) % 180
        hsv = np.array([[[hue, 200, 230]]], dtype=np.uint8)
        bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0][0]
        colors_bgr[cls_id] = (int(bgr[0]), int(bgr[1]), int(bgr[2]))
    return colors_bgr

CLASS_COLORS_BGR = generate_class_colors(num_classes)

def get_class_color(class_id: int) -> Tuple[int, int, int]:
    return CLASS_COLORS_BGR.get(class_id, (128, 128, 128))

console.print(f"[green]✔ Dynamic color palette initialized for {num_classes} classes.[/green]")

## 5. Core Inference Engine
Process images and output structured dictionaries including waste groups and explicit bounding box formats.

In [ ]:
# ==========================================
# Inference Logic
# ==========================================
def predict_image(
    model: YOLO,
    image_path: Union[str, Path],
    conf_threshold: float = 0.25,
    iou_threshold: float = 0.5
) -> Dict[str, Any]:
    """Run YOLO inference and return structured results."""
    image_path = Path(image_path)
    
    start_time = time.time()
    
    try:
        results = model.predict(
            source=str(image_path),
            conf=conf_threshold,
            iou=iou_threshold,
            verbose=False,
            imgsz=832
        )
    except Exception as e:
        return {
            'image_name': image_path.name,
            'status': 'error',
            'message': str(e)
        }
        
    result = results[0]
    boxes = result.boxes
    
    detections = []
    if boxes is not None:
        for i, box in enumerate(boxes):
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            
            # [x1, y1, x2, y2] - top-left, bottom-right
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().tolist()
            
            # [x, y, w, h] - center x, center y, width, height (normalized by Ultralytics or raw coords depending on xywh)
            # YOLO boxes.xywh is unnormalized pixel coordinates
            x_c, y_c, w, h = box.xywh[0].cpu().numpy().tolist()
            
            cls_name = class_names.get(cls_id, f"class_{cls_id}")
            waste_group = waste_groups.get(cls_id, "unknown")
            
            detections.append({
                'id': i,
                'class_id': cls_id,
                'class': cls_name,
                'waste_group': waste_group,
                'confidence': round(conf, 4),
                'bbox_xyxy': [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)],
                'bbox_xywh': [round(x_c, 1), round(y_c, 1), round(w, 1), round(h, 1)]
            })
            
    inf_time = (time.time() - start_time) * 1000
    
    return {
        'image_name': image_path.name,
        'status': 'success',
        'inference_time_ms': round(inf_time, 2),
        'total_objects': len(detections),
        'detections': detections
    }

## 6. Visualization
Draw bounding boxes using the dynamic color palette.

In [ ]:
def draw_boxes(image_path: Path, detections: List[Dict], output_path: Path = None):
    """Draw bounding boxes and labels on an image."""
    img = cv2.imread(str(image_path))
    if img is None:
        return None
        
    h, w = img.shape[:2]
    scale = min(w, h) / 640
    thickness = max(int(2 * scale), 1)
    font_scale = max(0.5 * scale, 0.4)
    
    for det in detections:
        cls_id = det['class_id']
        conf = det['confidence']
        x1, y1, x2, y2 = map(int, det['bbox_xyxy'])
        
        color = get_class_color(cls_id)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
        
        label = f"{det['class']} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
        
        cv2.rectangle(img, (x1, max(0, y1 - th - 5)), (x1 + tw, y1), color, -1)
        cv2.putText(img, label, (x1, y1 - 3), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)
        
    if output_path:
        cv2.imwrite(str(output_path), img)
        
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## 7. Run Batch Inference
Process all images in the test folder and save JSON/CSV/Image outputs.

In [ ]:
def process_folder(img_dir: Path, limit: int = 20):
    images = list(img_dir.glob('*.[jJ][pP][gG]')) + list(img_dir.glob('*.[pP][nN][gG]'))
    images = images[:limit] # Process a subset for demonstration
    
    all_results = []
    
    console.print(f"[cyan]Processing {len(images)} images...[/cyan]")
    for img_path in tqdm(images):
        res = predict_image(model, img_path)
        all_results.append(res)
        
        if res['status'] == 'success':
            # Save JSON
            with open(PRED_JSON_DIR / f"{img_path.stem}.json", 'w') as f:
                json.dump(res, f, indent=4)
                
            # Draw and save image
            draw_boxes(img_path, res['detections'], PRED_IMAGES_DIR / img_path.name)
            
    # Save CSV
    csv_rows = []
    for res in all_results:
        if res['status'] == 'success':
            for det in res['detections']:
                row = {
                    'image': res['image_name'],
                    'class_id': det['class_id'],
                    'class': det['class'],
                    'waste_group': det['waste_group'],
                    'confidence': det['confidence'],
                    'x1': det['bbox_xyxy'][0], 'y1': det['bbox_xyxy'][1],
                    'x2': det['bbox_xyxy'][2], 'y2': det['bbox_xyxy'][3]
                }
                csv_rows.append(row)
                
    if csv_rows:
        pd.DataFrame(csv_rows).to_csv(PRED_CSV_DIR / 'all_detections.csv', index=False)
        
    return all_results

results = process_folder(TEST_IMAGES, limit=10)
console.print(f"[green]✔ Batch inference complete. Outputs saved to {INFERENCE_DIR}[/green]")

In [ ]:
# Display a few processed images
processed_imgs = list(PRED_IMAGES_DIR.glob('*.[jJ][pP][gG]'))[:4]

if processed_imgs:
    fig, axes = plt.subplots(1, len(processed_imgs), figsize=(20, 5))
    if len(processed_imgs) == 1: axes = [axes]
    
    for i, img_path in enumerate(processed_imgs):
        img = cv2.imread(str(img_path))
        if img is not None:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            axes[i].set_title(img_path.name)
            axes[i].axis('off')
            
    plt.tight_layout()
    plt.show()

## 8. Summary
Inference engine is ready to be ported to FastAPI. Outputs conform to expected structured JSON format.